# Lab 1: How ACT predicts robot actions

**Goal:** Trace two camera images and an eight-value robot state through the saved ACT processor. Inspect one training loss and one predicted action chunk. Use a small queue to see how predictions can be combined across controller ticks.

ACT is a learned visuomotor controller. Images help it see the scene, and robot state helps it locate the arm. It predicts a short action sequence. The simulator controller then applies actions one at a time. A VLA such as SmolVLA also accepts a language instruction; this ACT checkpoint does not.

**Needs:** the local GPU, the complete ACT bundle, the selected training data, and about 2 GB of free work space. Run `.venv/bin/robot-lab data fetch --scope training` first. This notebook uses one training example and no final simulator states. It does not start a paid VM.

**Reading (20 minutes):** Read the architecture and loss parts of [the ACT paper](https://arxiv.org/abs/2304.13705). Predict whether the saved model returns one action or an action sequence before you run cell 4.

## 1. Set paths before model imports

Start Jupyter from the repository root with its `.venv` kernel. Keep the Hugging Face cache in `.local-data`. Choose one task; its checkpoint and training split must match.

In [ ]:
from pathlib import Path
import os
import json
import math

WORKSPACE = Path.cwd().resolve()
assert (WORKSPACE / "manifests" / "act_references.yaml").is_file(), "Start Jupyter from the repository root."
DATA_ROOT = WORKSPACE / ".local-data"
os.environ["HF_HUB_CACHE"] = str(DATA_ROOT / "hf" / "hub")
os.environ["HF_DATASETS_CACHE"] = str(DATA_ROOT / "hf" / "datasets")

import yaml
TASK = "bowl_plate"  # Or "bowl_stove".
references = yaml.safe_load((WORKSPACE / "manifests" / "act_references.yaml").read_text())
checkpoint = WORKSPACE / references["tasks"][TASK]["checkpoint_path"]
config = json.loads((checkpoint / "config.json").read_text())
print(json.dumps({key: config[key] for key in ("input_features", "output_features", "chunk_size", "n_action_steps", "use_vae", "latent_dim", "kl_weight", "temporal_ensemble_coeff")}, indent=2))
print(f"Equivalent live command: .venv/bin/robot-lab demo --task {TASK} --state-id 0 --output .local-data/runs/lab1-live")

The checkpoint expects two 256 × 256 camera views, an eight-value state, and seven-value actions. It predicts a 20-action chunk and executes 10 actions before it asks for another chunk. This model uses a conditional variational autoencoder (CVAE). During training, future actions help form a latent sample. During inference, the pinned LeRobot implementation uses a zero latent vector. The saved `temporal_ensemble_coeff` is null, so this reference does not use ACT temporal ensembling.

## 2. Inspect one training example

The frozen training split selects whole episodes. The dataset decoder returns camera tensors and a 20 × 7 target action chunk. `action_is_pad` marks any target positions after an episode ends.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_lab.training.act import _frozen_training_dataset

dataset = _frozen_training_dataset(TASK, WORKSPACE / "configs", DATA_ROOT)
sample = dataset[0]
print(f"Selected training frames: {len(dataset)}")
print({key: tuple(value.shape) for key, value in sample.items() if hasattr(value, "shape") and key in ("observation.images.image", "observation.images.image2", "observation.state", "action", "action_is_pad")})
print(f"Episode {int(sample['episode_index'])}, frame {int(sample['frame_index'])}, timestamp {float(sample['timestamp']):.3f} s")
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for axis, key, title in zip(axes, ("observation.images.image", "observation.images.image2"), ("Main camera", "Wrist camera"), strict=True):
    image = sample[key].permute(1, 2, 0).cpu().numpy()
    axis.imshow(np.clip(image, 0, 1))
    axis.set_title(title)
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Apply the saved processor and inspect loss

The processor uses the train-only state and action statistics saved with the checkpoint. It keeps the image path expected by the model. The training loss is an L1 action error plus `kl_weight × KL loss` when the CVAE is active. A single example is only a shape and finite-value check; it is not a quality score.

In [ ]:
import torch
from torch.utils.data._utils.collate import default_collate
from vla_lab.policy.bundle import load_act_bundle

policy, preprocessor, postprocessor, bundle = load_act_bundle(checkpoint)
batch = preprocessor(default_collate([sample]))
for key in ("observation.images.image", "observation.images.image2", "observation.state", "action", "action_is_pad"):
    print(key, tuple(batch[key].shape), batch[key].device)
assert tuple(batch["action"].shape) == (1, 20, 7)
torch.manual_seed(0)
policy.train()
with torch.no_grad():
    loss, parts = policy(batch)
print(f"L1: {parts['l1_loss']:.4f}; KL: {parts['kld_loss']:.4f}; total: {float(loss):.4f}")
assert math.isfinite(float(loss))
assert math.isclose(float(loss), parts["l1_loss"] + config["kl_weight"] * parts["kld_loss"], rel_tol=1e-4)

## 4. Predict one action chunk

Remove target actions from the batch. The model must now predict from observations alone. The saved checkpoint produces 20 normalized actions. The postprocessor maps them to controller values. The controller accepts values from −1 to 1; the live runner records how often values need clipping.

In [ ]:
policy.eval()
observations = {key: value for key, value in batch.items() if key not in ("action", "action_is_pad")}
with torch.no_grad():
    predicted = policy.predict_action_chunk(observations)
    physical_first = postprocessor(predicted[:, 0])
print("Predicted chunk:", tuple(predicted.shape))
print("First controller action:", np.round(physical_first.detach().cpu().numpy().reshape(-1), 3))
print("Values outside controller bounds:", int(((physical_first < -1) | (physical_first > 1)).sum()))
assert tuple(predicted.shape) == (1, 20, 7)

## 5. Align chunks in time

The live checkpoint normally executes 10 actions from each predicted chunk. The small queue below uses synthetic seven-value chunks so you can inspect indexing without changing model weights. At controller tick 11, the older chunk's second action is `0.2`; the newer chunk's first action is `0.6`. Equal-weight temporal ensembling gives `0.4`. The latest-only choice gives `0.6`.

In [ ]:
from vla_lab.contracts import ActionChunk, ActionSchema
from vla_lab.runtime.action_queue import ActionQueue

schema = ActionSchema.libero_relative()
def chunk(origin, values):
    actions = np.repeat(np.asarray(values, dtype=np.float32)[:, None], 7, axis=1)
    return ActionChunk("demo-episode", 0, f"request-{origin}", origin, origin, actions, schema, 50)

queue = ActionQueue("demo-episode", 0, TASK, ensemble_decay=0.0)
queue.push(chunk(10, [0.1, 0.2, 0.3]))
queue.push(chunk(11, [0.6, 0.7, 0.8]))
print("Latest-only at tick 11:", float(queue.action_at(11)[0]))
print("Ensemble at tick 11:", float(queue.action_at(11, ensemble=True)[0]))
assert np.isclose(queue.action_at(11, ensemble=True)[0], 0.4)

## Exercise cell

Change only `EXERCISE_LENGTH` to `5` after the reference path passes. Predict the number of model calls for the 74-step Lab 0 run. Then follow [the full exercise](exercise.md) and compare the two synthetic queue settings. The [solution](../../solutions/01-act.md) is separate.

In [ ]:
# EXERCISE: Change 10 to 5, then compare the result.
EXERCISE_LENGTH = 10
calls_for_74_steps = math.ceil(74 / EXERCISE_LENGTH)
print(f"Calls for 74 steps with execution length {EXERCISE_LENGTH}: {calls_for_74_steps}")

## Check and limits

You should see two camera views, a batch action shape `(1, 20, 7)`, a finite L1 and KL loss, a predicted chunk shape `(1, 20, 7)`, and the queue values `0.6` and `0.4`. A missing shard or AV1 decoder stops the data cell; a wrong GPU or bundle stops the model cells. Use `robot-lab doctor --live` and `robot-lab data fetch --scope training` to check those inputs.

This lab does not train a new model or estimate task success. A one-sample loss cannot select a checkpoint. Temporal ensembling here uses synthetic chunks; a later runtime lab tests it on a live policy. Keep simulator final states closed until the final comparison.